# Mimicry stress test (A18)

Can Sybil operators evade the published model by imitating ordinary users? A bounded stress test, not an attack
simulation: test Sybils' most important features are replaced with values taken from non-Sybil training wallets,
and the fixed model is scored again.

- **Model**: LightGBM, 3 seeds, parameters from `02`, trained here on the seed-42 tree split. The validation
  threshold is kept fixed: the defender does not adapt.
- **Attack order**: features ranked by mean |SHAP| over the test partition (`results/09_shap_importance.json`,
  column `All`).
- **Attacks** (on test Sybils only), for k ∈ {1, 2, 3, 5, 10, 20}, 5 random repetitions each:
  - **M1, marginal**: each of the top-k features drawn independently from the non-Sybil training rows (unique rows,
    before upsampling).
  - **M2, profile copy**: the top-k features copied jointly from one random non-Sybil training row per Sybil, which
    keeps their correlations.
  - **Cost-aware**: M1 and M2 restricted to the cheap features (ranked by SHAP), then to cheap plus structural.
- **Metrics**: test recall at the fixed threshold, F1, AP and the mean Sybil score (mean ± SD over repetitions), with
  the unattacked baseline.

`FEATURE_COST` below is **tentative, for the authors to confirm**.

**Caveats** (also stored in the results): replacing features ignores consistency between them (for example, counts
against time spans); it ignores the fees of the added activity; the defender does not retrain; the result measures
the fragility of the published model, not the cost or feasibility of a real attack.

**Results**: `results/18_mimicry_stress.json`.

## Setup

In [1]:
import os, platform, time, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
PLATFORM = dict(machine=platform.machine(), processor=platform.processor(), system=platform.system(),
                python=platform.python_version(), cpu_count=os.cpu_count(), n_jobs=sp.N_JOBS)
print('Platform:', PLATFORM)
prov = sp.provenance(['09_shap_importance'])
print(prov.to_string(index=False)); assert prov['dependencies_unchanged'].all()
DATA_DIR = './data'
FEATS = sp.FEATS
_, LGBM_P = sp.load_selected_params()

Code commit: 2ac94cb
Platform: {'machine': 'x86_64', 'processor': 'x86_64', 'system': 'Linux', 'python': '3.11.15', 'cpu_count': 4, 'n_jobs': 4}
          notebook  commit  dependencies_unchanged
09_shap_importance 38c3093                    True


## Feature cost classes (tentative, for the authors to confirm)

- **Cheap**: changeable now at fee cost: transaction counts and values; project, chain and contract counts; the
  LayerZero-to-Ethereum counts and values.
- **Time-bound**: needs earlier activity or knowledge of the snapshot date: time spans, first and last transaction
  times and blocks, and `gas_provision_block_number` (listed here although it belongs to the gas-provider family).
- **Structural**: needs a different funding topology: the rest of the gas provider, gas provision tree and
  provision chain families.

In [2]:
TIME_BOUND = ['l0_tx_time_span', 'earliest_l0_tx_time', 'latest_l0_tx_time', 'time_span_in', 'earliest_tx_block_in',
              'gas_provision_block_number', 'l0_to_eth_tx_time_span']
NETWORK = [f for k in ['Gas provider', 'Gas provision tree', 'Provision chain'] for f in sp.FEATURE_FAMILIES[k]]
STRUCTURAL = [f for f in NETWORK if f not in TIME_BOUND]
CHEAP = [f for f in FEATS if f not in TIME_BOUND and f not in STRUCTURAL]
FEATURE_COST = {**{f: 'cheap' for f in CHEAP}, **{f: 'time-bound' for f in TIME_BOUND}, **{f: 'structural' for f in STRUCTURAL}}
assert sorted(FEATURE_COST) == sorted(FEATS) and len(CHEAP) + len(TIME_BOUND) + len(STRUCTURAL) == len(FEATS)
print(f'cheap {len(CHEAP)}, time-bound {len(TIME_BOUND)}, structural {len(STRUCTURAL)}')
print('Cheap:', CHEAP)

shap = pd.DataFrame(sp.load_results('09_shap_importance')['mean_abs_shap'])
assert set(shap['feature']) == set(FEATS)
RANK = shap.sort_values('All', ascending=False)['feature'].tolist()
ORDERS = {'all features': RANK,
          'cheap only': [f for f in RANK if FEATURE_COST[f] == 'cheap'],
          'cheap + structural': [f for f in RANK if FEATURE_COST[f] in ('cheap', 'structural')]}
print(shap.sort_values('All', ascending=False).head(20).assign(cost=lambda t: t['feature'].map(FEATURE_COST))
      [['feature', 'All', 'family', 'cost']].round(4).to_string(index=False))

cheap 25, time-bound 7, structural 30
Cheap: ['min_tx_value_out', 'cex_in_count', 'l0_to_eth_avg_native_drop_usd', 'l0_to_eth_max_native_drop_usd', 'l0_avg_stargate_swap', 'indegree_per_block_in', 'l0_min_stargate_swap', 'num_transactions_in', 'n_l0_to_eth_source_contracts', 'n_l0_to_eth_projects', 'n_l0_to_eth_project_per_source_chain', 'n_l0_to_eth_txs', 'n_l0_projects', 'n_l0_to_eth_dest_contracts', 'l0_to_eth_min_stargate_swap', 'n_l0_source_chains', 'n_eth_interactions', 'max_tx_value_out', 'n_l0_source_contracts', 'min_tx_value_in', 'tx_value_per_block_out', 'n_l0_to_eth_source_chains', 'l0_to_eth_max_stargate_swap', 'n_l0_project_per_source_chain', 'l0_to_eth_avg_stargate_swap']
                            feature    All                 family       cost
              n_l0_source_contracts 0.9039 LayerZero transactions      cheap
                    l0_tx_time_span 0.9005 LayerZero transactions time-bound
                earliest_l0_tx_time 0.5431 LayerZero transactions time-bou

## Model (fixed; the defender does not adapt)

In [3]:
df, _, _ = sp.build_master_df(DATA_DIR, verbose=False)
S = sp.make_splits(df, FEATS, method='group', seed=sp.SEED)
m = sp.fit_lgbm(S, LGBM_P)
THR = sp.best_f1_threshold(S['y_val'], m['val'])

def score(X):
    return np.mean([mod.predict_proba(X)[:, 1] for mod in m['models']], axis=0)

assert np.allclose(score(S['X_test']), m['test'], rtol=0, atol=1e-12)
X_te, y_te = S['X_test'].copy(), S['y_test'].to_numpy()
is_syb = y_te == 1
donors = df.loc[S['idx_train']]
donors = donors.loc[donors['sybil'] == 0, FEATS].astype(np.float32).to_numpy()   # unique non-Sybil training rows
print(f'Threshold {THR:.4f}; {is_syb.sum():,} test Sybils; {len(donors):,} donor rows')

def metrics(p):
    yhat = (p >= THR).astype(int)
    return dict(recall=float(yhat[is_syb].mean()), f1=f1_score(y_te, yhat), ap=average_precision_score(y_te, p),
                mean_sybil_score=float(p[is_syb].mean()))
BASE = metrics(m['test'])
print('Unattacked:', {k: round(v, 4) for k, v in BASE.items()})

  LightGBM seed=42  rounds=315  49.8s


  LightGBM seed=123  rounds=298  46.8s


  LightGBM seed=456  rounds=301  49.0s


Threshold 0.5444; 5,463 test Sybils; 204,122 donor rows
Unattacked: {'recall': 0.7276, 'f1': 0.7149, 'ap': 0.7693, 'mean_sybil_score': 0.7186}


## Attacks

In [4]:
KS = [1, 2, 3, 5, 10, 20]
REPS = 5
col = {f: i for i, f in enumerate(FEATS)}
Xs = X_te.to_numpy()[is_syb]
rows = []
t0 = time.time()
for order_name, order in ORDERS.items():
    for k in KS:
        if k > len(order):
            continue
        idx = [col[f] for f in order[:k]]
        for attack in ['M1 marginal', 'M2 profile copy']:
            reps = []
            for rep in range(REPS):
                rng = np.random.default_rng(1000 * rep + k)
                A = Xs.copy()
                if attack == 'M1 marginal':
                    for j in idx:
                        A[:, j] = donors[rng.integers(0, len(donors), len(A)), j]
                else:
                    d = rng.integers(0, len(donors), len(A))
                    A[:, idx] = donors[d][:, idx]
                p = m['test'].copy()                       # non-Sybil scores are unchanged
                p[is_syb] = score(pd.DataFrame(A, columns=FEATS))
                reps.append(metrics(p))
            r = pd.DataFrame(reps)
            rows.append(dict(features=order_name, attack=attack, k=k, top_features=order[:k],
                             **{f'{c}_{a}': float(getattr(r[c], a)()) for c in r.columns for a in ('mean', 'std')}))
    print(f'{order_name} done ({time.time() - t0:.0f}s)', flush=True)
stress = pd.DataFrame(rows)

all features done (31s)


cheap only done (65s)


cheap + structural done (100s)


## Results: recall of test Sybils at the fixed threshold

In [5]:
print('Unattacked recall', round(BASE['recall'], 4), 'F1', round(BASE['f1'], 4), 'AP', round(BASE['ap'], 4))
tab = stress.assign(recall=lambda t: t['recall_mean'].map('{:.3f}'.format) + ' ± ' + t['recall_std'].map('{:.3f}'.format))
print(tab.pivot_table(index=['features', 'attack'], columns='k', values='recall', aggfunc='first').to_string())
print()
print(stress[['features', 'attack', 'k', 'recall_mean', 'f1_mean', 'ap_mean', 'mean_sybil_score_mean']].round(4).to_string(index=False))

Unattacked recall 0.7276 F1 0.7149 AP 0.7693
k                                              1              2              3              5              10             20
features           attack                                                                                                   
all features       M1 marginal      0.384 ± 0.004  0.158 ± 0.003  0.056 ± 0.002  0.012 ± 0.001  0.001 ± 0.000  0.001 ± 0.000
                   M2 profile copy  0.384 ± 0.004  0.232 ± 0.006  0.126 ± 0.003  0.039 ± 0.003  0.006 ± 0.001  0.005 ± 0.001
cheap + structural M1 marginal      0.384 ± 0.004  0.201 ± 0.004  0.111 ± 0.001  0.021 ± 0.002  0.004 ± 0.001  0.000 ± 0.000
                   M2 profile copy  0.384 ± 0.004  0.215 ± 0.002  0.127 ± 0.003  0.038 ± 0.000  0.010 ± 0.002  0.003 ± 0.001
cheap only         M1 marginal      0.384 ± 0.004  0.201 ± 0.004  0.111 ± 0.001  0.021 ± 0.002  0.004 ± 0.001  0.002 ± 0.001
                   M2 profile copy  0.384 ± 0.004  0.215 ± 0.002  0.127 ± 0.003 

## Save results

In [6]:
sp.save_results([], '18_mimicry_stress', extra=dict(
    params=LGBM_P, threshold=THR, baseline=BASE, ks=KS, repetitions=REPS,
    feature_cost=FEATURE_COST, feature_cost_status='tentative, for the authors to confirm',
    attack_orders={k: v for k, v in ORDERS.items()}, results=stress.to_dict('records'),
    caveats=['Replacing features ignores consistency between them (for example, counts against time spans).',
             'It ignores the fees of the added activity.',
             'The defender does not retrain; the validation threshold is fixed.',
             'The result measures the fragility of the published model, not the cost or feasibility of a real attack.'],
    platform=PLATFORM))

Saved results/18_mimicry_stress.json
